* Classification predicts a category, such as survived or not, spam or not, customer churns or not.

* Binary has 2 classes (0 or 1).

* Multiclass has 3 or more (penguin species).

* Why not just use linear regression on a 0/1 label? It can output values like −0.19 or 1.10, which are not probabilities. We need an output that stays between 0 and 1.

* The sigmoid function squeezes any number z into the range 0 to 1:
        - sigmoid(z) = 1 / (1 + e^(−z))
        - z = 0 gives 0.5. A large positive z gives close to 1. A large negative z gives close to 0.

* Logistic regression is a linear model followed by the sigmoid:
        - z = x · w + b (exactly the weighted sum from Day 6)
        - p = sigmoid(z) is the probability that the class is 1.

* The threshold: predict class 1 if p ≥ 0.5, otherwise class 0. You can move the threshold (Hour 2 shows why).

* Reading a weight: exp(w) is the factor by which the odds (p / (1 − p)) change when that feature goes up by 1, with the others fixed. For example, exp(w) = 0.35 means the odds are multiplied by 0.35.

* Training: the weights are found by minimising log loss (a penalty that is large when the model is confident and wrong), using gradient descent. It is the same idea as Day 6 with a different loss.

* Multiclass: the model gives one probability per class, and the probabilities in each row add up to 1.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
# 1. the sigmoid: turns ANY number z into a probability between 0 and 1
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print([round(float(sigmoid(z)), 3) for z in (-4, -2, 0, 2, 4)])

[0.018, 0.119, 0.5, 0.881, 0.982]


In [3]:
# 2. Titanic, same data and split as Day 7
df = sns.load_dataset("Titanic")
data = df[["survived", "pclass", "sex", "age", "fare", "sibsp", "parch"]].copy()
data["is_female"] = (data["sex"] == "female").astype(int)
X = data[["pclass", "is_female", "age", "fare", "sibsp", "parch"]]
y = data["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(max_iter=1000))
])
pipeline.fit(X_train, y_train)
print(round(pipeline.score(X_test, y_test), 4))


0.8045


In [4]:
# 3. the learned weights and what they mean
lr = pipeline.named_steps["model"]
print(pd.Series(lr.coef_[0], index=X.columns).round(3).to_dict())
print(lr.intercept_.round(3), lr.classes_)
print(pd.Series(np.exp(lr.coef_[0]), index=X.columns).round(3).to_dict())

{'pclass': -1.051, 'is_female': 2.613, 'age': -0.039, 'fare': 0.003, 'sibsp': -0.268, 'parch': -0.102}
[2.066] [0 1]
{'pclass': 0.35, 'is_female': 13.644, 'age': 0.962, 'fare': 1.003, 'sibsp': 0.765, 'parch': 0.903}


In [5]:
# 4. ONE passenger by hand: z = x . w + b, then sigmoid(z)
row = X_test.iloc[[0]]
print(row.to_dict("records"))

x_row = pipeline.named_steps["imputer"].transform(row)[0]
z = x_row @ lr.coef_[0] + lr.intercept_[0]
print(round(float(z), 4), round(float(sigmoid(z)), 4))
print(pipeline.predict_proba(row).round(4), pipeline.predict(row), y_test.iloc[0])

[{'pclass': 3, 'is_female': 0, 'age': 24.0, 'fare': 24.15, 'sibsp': 2, 'parch': 0}]
-2.4709 0.0779
[[0.9221 0.0779]] [0] 0


In [6]:
# 5. probabilities for all test passengers, and the threshold
proba = pipeline.predict_proba(X_test)
print(proba.shape, proba[:3].sum(axis=1))

p_survive = proba[:, 1]
for t in (0.3, 0.5, 0.7):
    pred = (p_survive >= t).astype(int)
    print(t, pred.sum(), round((pred == y_test.values).mean(), 4))

(179, 2) [1. 1. 1.]
0.3 82 0.7709
0.5 64 0.8045
0.7 33 0.7765


In [10]:
# 6. why not LinearRegression for a 0/1 label?
X_tr_f = X_train.fillna({"age": X_train["age"].median()})
X_te_f = X_test.fillna({"age": X_train["age"].median()})
lin = LinearRegression().fit(X_tr_f, y_train)
p_lin = lin.predict(X_te_f)
print(round(p_lin.min(), 3), round(p_lin.max(), 3), (p_lin < 0).sum(), (p_lin > 1).sum())

-0.192 1.097 4 6


In [13]:
# 7. more than two classes: penguin species
p = sns.load_dataset("penguins").dropna()
Xp = p[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]
yp = p["species"]
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    Xp, yp, test_size=0.2, random_state=42, stratify=yp)
mp = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
mp.fit(Xp_tr, yp_tr)
print(Xp_tr.shape, Xp_te.shape, round(mp.score(Xp_te, yp_te), 4))
pr = mp.predict_proba(Xp_te)
print(mp.classes_, pr.shape)
print(pr[:2].round(3), pr[:2].sum(axis=1))
print(mp.predict(Xp_te)[:3], yp_te.iloc[:3].tolist())

(266, 4) (67, 4) 1.0
['Adelie' 'Chinstrap' 'Gentoo'] (67, 3)
[[0.    0.021 0.979]
 [0.092 0.875 0.033]] [1. 1.]
['Gentoo' 'Chinstrap' 'Adelie'] ['Gentoo', 'Chinstrap', 'Adelie']
